In [8]:
import torch
import numpy as np
import os
from tqdm.notebook import tqdm
from utils import assert_equal, load_pickle, save_pickle
from sklearn.metrics import roc_auc_score, roc_curve
from matplotlib import pyplot as plt


## Model configs & cached loading

In [3]:
from calc_rmse import get_dataloader, get_prediction_fn
from types import SimpleNamespace

MODELS = {
    'nn_24880': dict(eval_type='nn', version=24880),
    # 'nn_28114': dict(eval_type='nn', version=28114),
    'pysr_24880_11003_26': dict(eval_type='pysr', version=24880, pysr_version=11003, pysr_model_selection='26'),
    'petit': dict(eval_type='petit'),
    'pure_sr_72420_41': dict(eval_type='pure_sr', pysr_version=72420, pysr_model_selection='41'),
    'pysr_28114_41564_45': dict(eval_type='pysr', version=28114, pysr_version=41564, pysr_model_selection='45'),
}

# Whether a model produces full [mean, std] NN output
def _is_nn_model(cfg):
    return cfg['eval_type'] in ('nn', 'pysr')


def _cache_path(model_key, dataset):
    return f'pickles/cache_full_{model_key}_{dataset}.pkl'


def load_model(model_key, dataset='val'):
    """Load model + data with per-model disk caching.

    For NN models (nn, pysr): caches (xs, truths, preds) as full tensors
        where preds is [N, 2] (mean, std).
    For non-NN models (petit, pure_sr): caches (xs, truths, preds) where
        preds is a 1-D numpy array of point predictions.
    """
    global xs, truths, preds, current_model, current_dataset

    path = _cache_path(model_key, dataset)
    if os.path.exists(path):
        cached = load_pickle(path)
        xs, truths, preds = cached['xs'], cached['truths'], cached['preds']
        current_model = model_key
        current_dataset = dataset
        # print('Loaded from cache:', path)
        return

    cfg = MODELS[model_key]
    is_petit = cfg['eval_type'] == 'petit'
    dataloader = get_dataloader(dataset, petit=is_petit)
    args = SimpleNamespace(dataset=dataset, **cfg)
    predict_fn = get_prediction_fn(args)

    if _is_nn_model(cfg):
        # NN-style: predict_fn wraps model forward, returns numpy of just means
        # We need full [mean, std] output, so load model directly
        import spock_reg_model
        if cfg['eval_type'] == 'pysr':
            m = spock_reg_model.load_with_pysr_f2(
                version=cfg['version'],
                pysr_version=cfg['pysr_version'],
                pysr_model_selection=cfg['pysr_model_selection'],
            )
        else:
            m = spock_reg_model.load(version=cfg['version'])
        m.eval()
        m.cuda()

        def nn_predict(x):
            return m(x.cuda(), noisy_val=False, deterministic=True).cpu().detach()

        _xs, _truths, _preds = [], [], []
        for x, y in tqdm(dataloader):
            _xs.append(x)
            _truths.append(y)
            _preds.append(nn_predict(x))

        xs = torch.cat(_xs, dim=0)
        truths = torch.cat(_truths, dim=0)
        preds = torch.cat(_preds, dim=0)
        assert_equal(truths.shape, preds.shape)
    else:
        # Non-NN: predict_fn returns 1-D numpy array of point predictions
        _xs, _truths, _preds = [], [], []
        for x, y in tqdm(dataloader):
            _xs.append(x)
            _truths.append(y.numpy())
            _preds.append(predict_fn(x))

        xs = torch.cat(_xs, dim=0)
        truths = np.concatenate(_truths)
        preds = np.concatenate(_preds)
        preds[np.isnan(preds)] = 4

    save_pickle({'xs': xs, 'truths': truths, 'preds': preds}, path)
    print(f'Cached {model_key}/{dataset} to {path}')

    current_model = model_key
    current_dataset = dataset

/home/sca63/.conda/envs/pysr/lib/python3.10/site-packages/juliacall/__init__.py:60: UserWarning: torch was imported before juliacall. This may cause a segfault. To avoid this, import juliacall before importing torch. For updates, see https://github.com/pytorch/pytorch/issues/78829.
  warnings.warn(


Detected Jupyter notebook. Loading juliacall extension. Set `PYSR_AUTOLOAD_EXTENSIONS=no` to disable.


In [4]:
# Preload all models and datasets
for model_key in MODELS:
    for dataset in ['train', 'val', 'test', 'random']:
        load_model(model_key, dataset=dataset)

print('All models and data loaded.')

All models and data loaded.


In [5]:
def safe_log_erf(x):
    base_mask = x < -1
    value_giving_zero = torch.zeros_like(x, device=x.device)
    x_under = torch.where(base_mask, x, value_giving_zero)
    x_over = torch.where(~base_mask, x, value_giving_zero)

    f_under = lambda x: (
         0.485660082730562*x + 0.643278438654541*torch.exp(x) +
         0.00200084619923262*x**3 - 0.643250926022749 - 0.955350621183745*x**2
    )
    f_over = lambda x: torch.log(1.0+torch.erf(x))

    return f_under(x_under) + f_over(x_over)


def lossfnc(testy, y):
    mu = testy[:, [0]]
    std = testy[:, [1]]

    var = std**2
    t_greater_9 = y >= 9

    regression_loss = -(y - mu)**2/(2*var)
    regression_loss += -torch.log(std)

    regression_loss += -safe_log_erf(
                (mu - 4)/(torch.sqrt(2*var))
            )

    classifier_loss = (
        safe_log_erf((mu - 9) / torch.sqrt(2*var)) -
        safe_log_erf((mu - 4) / torch.sqrt(2*var))
    )

    safe_regression_loss = torch.where(
            ~torch.isfinite(regression_loss),
            -torch.ones_like(regression_loss)*100,
            regression_loss)
    safe_classifier_loss = torch.where(
            ~torch.isfinite(classifier_loss),
            -torch.ones_like(classifier_loss)*100,
            classifier_loss)

    total_loss = (
        safe_regression_loss * (~t_greater_9) +
        safe_classifier_loss * ( t_greater_9)
    )

    total_regression = safe_regression_loss * (~t_greater_9)
    total_classifier = safe_classifier_loss * ( t_greater_9)
    total_regression = -(total_regression.sum() / len(y)).item()
    total_classifier = -(total_classifier.sum() / len(y)).item()
    avg_std = std.mean().item()

    total_loss = total_regression + total_classifier

    return total_loss, total_regression, total_classifier, avg_std

## Run loss + metrics across all models

In [10]:
SPLITS = ['train', 'val', 'test', 'random']

def _get_clipped_preds_and_truths():
    """Extract clipped predictions and averaged truths, handling both NN and non-NN models."""
    cfg = MODELS[current_model]
    if _is_nn_model(cfg):
        p = np.clip(preds[:, 0].numpy(), 4, 9)
        t = np.average(truths.numpy(), axis=1)
    else:
        p = np.clip(preds, 4, 9)
        t = np.average(truths, axis=1)
    return p, t


def calculate_metrics_from_cache():
    """Calculate metrics from the cached preds and truths.

    Positive = predicting unstable (pred < 9).
      FPR = predict unstable when truly stable (T >= 9)
      FNR = predict stable when truly unstable (T < 9)
    """
    p, t = _get_clipped_preds_and_truths()

    pred_stable = p >= 9
    pred_unstable = ~pred_stable
    true_stable = t >= 9
    true_unstable = ~true_stable

    tp = np.sum(pred_unstable & true_unstable)
    tn = np.sum(pred_stable & true_stable)
    fp = np.sum(pred_unstable & true_stable)
    fn = np.sum(pred_stable & true_unstable)
    n_stable = np.sum(true_stable)
    n_unstable = np.sum(true_unstable)

    rmse = np.sqrt(np.mean((t[true_unstable] - p[true_unstable])**2))
    acc = np.mean(pred_stable == true_stable)
    stable_only_acc = _safe_rate(tn, n_stable)
    unstable_recall = _safe_rate(tp, n_unstable)
    balanced_acc = 0.5 * (stable_only_acc + unstable_recall)
    unstable_score = -p
    roc_auc = _roc_auc_or_nan(true_unstable, unstable_score)
    roc_path = _save_roc_plot(true_unstable, unstable_score, roc_auc)
    bias = np.mean(p - t)
    fpr = _safe_rate(fp, n_stable)
    fnr = _safe_rate(fn, n_unstable)
    precision_unstable = _safe_rate(tp, tp + fp)
    mcc_denom = np.sqrt(float(tp + fp) * float(tp + fn) * float(tn + fp) * float(tn + fn))
    mcc = ((tp * tn) - (fp * fn)) / mcc_denom if mcc_denom else 0.0

    metrics = dict(
        rmse=rmse,
        acc=acc,
        balanced_acc=balanced_acc,
        roc_auc=roc_auc,
        auc=roc_auc,
        bias=bias,
        stable_only_acc=stable_only_acc,
        fpr=fpr,
        fnr=fnr,
        precision_unstable=precision_unstable,
        mcc=mcc,
        roc_path=roc_path,
    )
    for target_fpr in FIXED_FPR_TARGETS:
        key = f'fnr_at_fpr_{int(round(target_fpr * 100)):03d}'
        metrics[key] = _fnr_at_fixed_fpr(true_unstable, unstable_score, target_fpr)
    return metrics


def _safe_rate(numerator, denominator):
    return numerator / denominator if denominator else np.nan


def _roc_auc_or_nan(labels, scores):
    return roc_auc_score(labels, scores) if len(np.unique(labels)) == 2 else np.nan


def _save_roc_plot(labels, scores, roc_auc):
    path = f'plots/roc/{current_model}_{current_dataset}.png'
    os.makedirs(os.path.dirname(path), exist_ok=True)

    plt.switch_backend('agg')
    fig, ax = plt.subplots()
    if len(np.unique(labels)) == 2:
        fprs, tprs, _ = roc_curve(labels, scores)
        label = f'AUC = {roc_auc:.4f}' if not np.isnan(roc_auc) else 'AUC = nan'
        ax.plot(fprs, tprs, label=label)
    else:
        ax.text(0.5, 0.5, 'ROC undefined: one class present', ha='center', va='center')

    ax.plot([0, 1], [0, 1], 'k--', label='Random')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curve - {current_model} ({current_dataset})')
    ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved ROC plot to {path}')
    return path


def _fnr_at_fixed_fpr(labels, scores, target_fpr):
    if len(np.unique(labels)) < 2:
        return np.nan
    fprs, tprs, _ = roc_curve(labels, scores)
    unique_fprs = np.unique(fprs)
    best_tprs = np.array([np.max(tprs[fprs == fpr]) for fpr in unique_fprs])
    target_tpr = np.interp(target_fpr, unique_fprs, best_tprs)
    return 1.0 - target_tpr


FIXED_FPR_TARGETS = (0.05, 0.10, 0.20)
METRIC_COLUMNS = [
    ('rmse', 'RMSE'),
    ('acc', 'Acc'),
    ('balanced_acc', 'BalancedAcc'),
    ('roc_auc', 'ROC-AUC'),
    ('fpr', 'FPR'),
    ('fnr', 'FNR'),
    ('fnr_at_fpr_005', 'FNR@FPR=0.05'),
    ('fnr_at_fpr_010', 'FNR@FPR=0.10'),
    ('fnr_at_fpr_020', 'FNR@FPR=0.20'),
    ('precision_unstable', 'PrecisionUnstable'),
    ('mcc', 'MCC'),
    ('bias', 'Bias'),
]


def _format_metric(value):
    if isinstance(value, (float, np.floating)) and np.isnan(value):
        return 'nan'
    return f'{value:.4f}'


def _print_metrics_table(table_rows):
    headers = ['method'] + [label for _, label in METRIC_COLUMNS]
    rows = [[name] + [_format_metric(metrics[key]) for key, _ in METRIC_COLUMNS]
            for name, metrics in table_rows]
    widths = [max(len(str(row[i])) for row in [headers] + rows) for i in range(len(headers))]

    def format_row(row):
        return '\t'.join(str(cell).ljust(widths[i]) for i, cell in enumerate(row))

    print(format_row(headers))
    for row in rows:
        print(format_row(row))


for split in ('test', 'random'):
    table_rows = []
    for name in MODELS:
        load_model(name, dataset=split)
        table_rows.append((name, calculate_metrics_from_cache()))

    print(f'=== {split} ===')
    _print_metrics_table(table_rows)
    print()


Saved ROC plot to plots/roc/nn_24880_test.png
Saved ROC plot to plots/roc/pysr_24880_11003_26_test.png
Saved ROC plot to plots/roc/petit_test.png
Saved ROC plot to plots/roc/pure_sr_72420_41_test.png
Saved ROC plot to plots/roc/pysr_28114_41564_45_test.png
=== test ===
method             	RMSE  	Acc   	BalancedAcc	ROC-AUC	FPR   	FNR   	FNR@FPR=0.05	FNR@FPR=0.10	FNR@FPR=0.20	PrecisionUnstable	MCC   	Bias   
nn_24880           	1.1196	0.8748	0.8260     	0.9146 	0.2756	0.0725	0.2975      	0.1969      	0.1077      	0.9056           	0.6677	-0.1075
pysr_24880_11003_26	1.3393	0.8119	0.6837     	0.8654 	0.5829	0.0497	0.4936      	0.3533      	0.2207      	0.8230           	0.4575	-0.0076
petit              	3.1291	0.3118	0.5333     	0.5333 	0.0059	0.9275	0.8864      	0.8398      	0.7465      	0.9720           	0.1277	2.0651 
pure_sr_72420_41   	1.4097	0.7403	0.5000     	0.7434 	1.0000	0.0000	0.7630      	0.6466      	0.4749      	0.7403           	0.0000	-0.0557
pysr_28114_41564_45	1.2191	0.7